# Patient Forecasting — Model Training & Evaluation

**Models:** XGBoost Regressor (LOS) · SARIMAX (Patient Arrivals)

Each process has its own cell for clarity.

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error

from statsmodels.tsa.statespace.sarimax import SARIMAX

print('✅ Imports OK')

## 2. Generate Synthetic Patient Data (LOS)

In [ ]:
def create_synthetic_patient_data(num_samples=5000):
    """Same logic as production los_train.py"""
    np.random.seed(42)

    age              = np.random.randint(18, 90, num_samples)
    gender           = np.random.choice(['M', 'F'], num_samples)
    admission_type   = np.random.choice(
                           ['emergency', 'elective', 'urgent'],
                           num_samples, p=[0.4, 0.4, 0.2])
    condition_severity = np.random.randint(1, 6, num_samples)
    num_diagnoses    = np.random.randint(1, 10, num_samples)
    num_procedures   = np.random.randint(0, 5, num_samples)
    insurance_type   = np.random.choice(
                           ['Medicare', 'Medicaid', 'Private', 'Self-Pay'],
                           num_samples)

    base_stay        = 2
    age_factor       = (age - 18) / 30
    severity_factor  = condition_severity * 1.5
    procedure_factor = num_procedures * 2.5
    admit_factor     = np.where(admission_type == 'emergency', 3,
                       np.where(admission_type == 'urgent', 2, 0))
    noise            = np.random.normal(0, 1.5, num_samples)

    los = base_stay + age_factor + severity_factor + procedure_factor + admit_factor + noise
    los = np.clip(np.round(los), 1, 30).astype(int)

    return pd.DataFrame({
        'age': age, 'gender': gender,
        'admission_type': admission_type,
        'condition_severity': condition_severity,
        'num_diagnoses': num_diagnoses,
        'num_procedures': num_procedures,
        'insurance_type': insurance_type,
        'length_of_stay': los
    })

print('✅ Function defined')

In [ ]:
patient_df = create_synthetic_patient_data(5000)
print(f'Dataset shape: {patient_df.shape}')
print(f"LOS range: {patient_df['length_of_stay'].min()}–{patient_df['length_of_stay'].max()} days")
patient_df.head()

## 3. Encode Categorical Features

In [ ]:
df = patient_df.copy()

le_gender    = LabelEncoder()
le_admission = LabelEncoder()
le_insurance = LabelEncoder()

df['gender_enc']         = le_gender.fit_transform(df['gender'])
df['admission_type_enc'] = le_admission.fit_transform(df['admission_type'])
df['insurance_type_enc'] = le_insurance.fit_transform(df['insurance_type'])

print('Encoded classes:')
print('  gender      :', list(le_gender.classes_))
print('  admission   :', list(le_admission.classes_))
print('  insurance   :', list(le_insurance.classes_))

## 4. Train / Test Split

In [ ]:
FEATURES = ['age', 'gender_enc', 'admission_type_enc',
            'condition_severity', 'num_diagnoses',
            'num_procedures', 'insurance_type_enc']
FEATURE_LABELS = ['Age', 'Gender', 'Admission Type',
                  'Condition Severity', 'No. Diagnoses',
                  'No. Procedures', 'Insurance Type']

X = df[FEATURES]
y = df['length_of_stay']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train size : {X_train.shape[0]} samples')
print(f'Test  size : {X_test.shape[0]}  samples')

## 5. Train XGBoost Regressor (LOS)

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
xgb_model.fit(X_train, y_train)
print('✅ XGBoost training complete')

## 6. Evaluate LOS Model

In [ ]:
y_pred = xgb_model.predict(X_test)
r2     = r2_score(y_test, y_pred)
mae    = mean_absolute_error(y_test, y_pred)

print(f'XGBoost LOS Results')
print(f'  R²  = {r2:.4f}  (1.0 = perfect)')
print(f'  MAE = {mae:.2f} days')

## 7. Graph — Actual vs Predicted LOS

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(y_test, y_pred, alpha=0.3, color='steelblue', edgecolors='none', s=15)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect Prediction')

ax.set_xlabel('Actual LOS (days)')
ax.set_ylabel('Predicted LOS (days)')
ax.set_title(f'Actual vs Predicted — LOS  (R²={r2:.3f}, MAE={mae:.2f}d)')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 8. Graph — Feature Importances

In [ ]:
importances = xgb_model.feature_importances_
sorted_idx    = np.argsort(importances)
sorted_labels = [FEATURE_LABELS[i] for i in sorted_idx]
sorted_vals   = importances[sorted_idx]

colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(sorted_vals)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(sorted_labels, sorted_vals, color=colors, edgecolor='k', linewidth=0.4)
ax.set_xlabel('Importance Score')
ax.set_title('XGBoost Feature Importances — LOS Prediction')
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 9. Graph — Residuals Distribution

In [ ]:
residuals = y_test.values - y_pred

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(residuals, bins=40, color='cornflowerblue', edgecolor='white', linewidth=0.5)
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero Error')
ax.set_xlabel('Residual (Actual − Predicted, days)')
ax.set_ylabel('Count')
ax.set_title('Residuals Distribution — LOS')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 10. Generate Synthetic Arrival Data (SARIMAX)

In [ ]:
def create_synthetic_arrival_data(num_days=730):
    """Same logic as production arrival_forecast_train.py"""
    np.random.seed(42)
    dates  = pd.date_range(end=pd.Timestamp.today(), periods=num_days, freq='D')
    days   = np.arange(num_days)

    yearly_seasonality = np.sin(2 * np.pi * days / 365) * 40

    day_of_week    = dates.dayofweek
    weekly_pattern = np.zeros(num_days)
    weekly_pattern[day_of_week == 0] =  30
    weekly_pattern[day_of_week == 1] =  20
    weekly_pattern[day_of_week == 5] = -40
    weekly_pattern[day_of_week == 6] = -50

    noise    = np.random.normal(0, 15, num_days)
    arrivals = 200 + yearly_seasonality + weekly_pattern + noise
    arrivals = np.clip(np.round(arrivals), 50, None).astype(int)

    return pd.DataFrame({'ds': dates, 'y': arrivals}).set_index('ds')

print('✅ Function defined')

In [ ]:
arrival_df = create_synthetic_arrival_data(730)
print(f'Arrival dataset: {arrival_df.shape}')
print(f"Date range : {arrival_df.index[0].date()} to {arrival_df.index[-1].date()}")
arrival_df.tail()

## 11. Split Arrival Data for Evaluation

In [ ]:
HOLDOUT = 30   # hold out last 30 days as test set

train_series = arrival_df['y'].iloc[:-HOLDOUT]
test_series  = arrival_df['y'].iloc[-HOLDOUT:]

print(f'Train: {len(train_series)} days')
print(f'Test (hold-out): {len(test_series)} days')

## 12. Train SARIMAX Model

In [ ]:
print('Training SARIMAX(1,1,1)(1,1,1,7) … (~1–2 min)')

sarimax_model   = SARIMAX(
    train_series,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarimax_results = sarimax_model.fit(disp=False)
print(f'✅ SARIMAX fit complete  — AIC = {sarimax_results.aic:.1f}')

## 13. Evaluate SARIMAX

In [ ]:
forecast  = sarimax_results.forecast(steps=HOLDOUT)
mae_arr   = mean_absolute_error(test_series, forecast)

print(f'SARIMAX Arrival Forecast')
print(f'  AIC = {sarimax_results.aic:.1f}')
print(f'  MAE (hold-out {HOLDOUT}d) = {mae_arr:.1f} patients/day')

## 14. Graph — Patient Arrival Forecast

In [ ]:
history_window = arrival_df['y'].iloc[-90:-HOLDOUT]   # 90-day context

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(history_window.index, history_window.values,
        color='steelblue', linewidth=1.2, label='Historical (90d)')
ax.plot(test_series.index, test_series.values,
        color='darkorange', linewidth=1.5, label='Actual (hold-out)')
ax.plot(test_series.index, forecast.values,
        color='red', linewidth=1.5, linestyle='--', label='SARIMAX Forecast')

ax.set_xlabel('Date')
ax.set_ylabel('Patient Arrivals / Day')
ax.set_title(f'Patient Arrival Forecast — SARIMAX  (MAE={mae_arr:.1f} patients/day)')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 15. Graph — Daily Forecast Error

In [ ]:
errors = test_series.values - forecast.values
colors_err = ['tomato' if e < 0 else 'mediumseagreen' for e in errors]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(HOLDOUT), errors, color=colors_err, edgecolor='white', linewidth=0.4)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Day in Forecast Window')
ax.set_ylabel('Error (Actual − Forecast)')
ax.set_title('Daily Forecast Error — SARIMAX Arrival')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Summary

| # | Model | Task | Metric |
|---|---|---|---|
| 1 | **XGBoost Regressor** | Length of Stay | R², MAE |
| 2 | **SARIMAX** | Patient Arrivals | AIC, MAE |

Both models match what is served by the FastAPI backend.